In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

train = pd.read_csv('train.csv')
layout = pd.read_csv('layout_info.csv')

groups = train['layout_id'].copy()

train = train.merge(layout, on='layout_id', how='left')

train = pd.get_dummies(train, columns=['layout_type'])

y = train['avg_delay_minutes_next_30m']
X = train.drop(['ID', 'layout_id', 'scenario_id', 'avg_delay_minutes_next_30m'], axis=1)

In [5]:
lgb_params = {
    'n_estimators': 1000, 'learning_rate': 0.0100, 'num_leaves': 140,
    'max_depth': 8, 'min_child_samples': 25, 'subsample': 0.9608,
    'colsample_bytree': 0.6607, 'reg_alpha': 1.2338, 'reg_lambda': 7.6823,
    'random_state': 42, 'verbose': -1
}

In [6]:
X['robot_density'] = X['robot_total'] / X['floor_area_sqm']
X['robot_per_charger'] = X['robot_active'] / (X['charger_count'] + 1)
X['order_per_packstation'] = X['order_inflow_15m'] / (X['pack_station_count'] + 1)
X['order_per_robot'] = X['order_inflow_15m'] / (X['robot_total'] + 1)

In [7]:
gkf = GroupKFold(n_splits=5)
maes = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    m = LGBMRegressor(**lgb_params)
    m.fit(X_tr, np.log1p(y_tr))
    pred = np.expm1(m.predict(X_va))
    mae = mean_absolute_error(y_va, pred)
    maes.append(mae)
    print(f'Fold {fold} MAE: {mae:.3f}')

Fold 1 MAE: 9.202
Fold 2 MAE: 10.627
Fold 3 MAE: 9.052
Fold 4 MAE: 7.840
Fold 5 MAE: 9.212


In [8]:
print(f'\n[layout 특성 추가] 평균 MAE: {np.mean(maes):.3f}')
print(f'(비교) layout 없을 때: 9.431')


[layout 특성 추가] 평균 MAE: 9.187
(비교) layout 없을 때: 9.431


In [9]:
import pandas as pd

importance2 = pd.Series(m.feature_importances_, index=X.columns)
print(importance2.sort_values(ascending=False).head(20))

sku_concentration        4982
pack_station_count       4424
order_per_packstation    4277
avg_trip_distance        4220
layout_compactness       3891
manual_override_ratio    3220
zone_dispersion          3159
avg_items_per_order      2621
aisle_width_avg          2499
robot_density            2269
floor_area_sqm           2249
pack_utilization         2237
one_way_ratio            2150
ceiling_height_m         2124
fire_sprinkler_count     2111
robot_total              2063
battery_mean             2042
building_age_years       1978
congestion_score         1971
intersection_count       1868
dtype: int32


In [10]:
import pandas as pd

train = pd.read_csv('train.csv')

# 컬럼별 결측 비율
miss = (train.isnull().sum() / len(train)).sort_values(ascending=False)
print("결측 비율 상위 15개:")
print(miss.head(15))

print("\n결측 있는 컬럼 수:", (miss > 0).sum())
print("결측 비율 평균:", miss[miss > 0].mean())

결측 비율 상위 15개:
avg_recovery_time            0.130116
congestion_score             0.129000
avg_charge_wait              0.122784
battery_mean                 0.121280
charge_efficiency_pct        0.120208
battery_cycle_count_avg      0.119820
fleet_age_months_avg         0.119812
robot_calibration_score      0.119776
unique_sku_15m               0.119696
staging_area_util            0.119568
humidity_pct                 0.119524
barcode_read_success_rate    0.119448
inventory_turnover_rate      0.119412
worker_avg_tenure_months     0.119216
shift_hour                   0.119188
dtype: float64

결측 있는 컬럼 수: 86
결측 비율 평균: 0.11878172093023255


In [11]:
import numpy as np

# 각 행의 결측 개수
train['missing_count'] = train.isnull().sum(axis=1)

# 결측 개수 분포
print("행당 결측 개수 분포:")
print(train['missing_count'].describe())

# 결측 개수와 타겟(지연)의 상관관계
corr = train['missing_count'].corr(train['avg_delay_minutes_next_30m'])
print(f"\n결측 개수 vs 지연 상관계수: {corr:.3f}")

행당 결측 개수 분포:
count    250000.000000
mean         10.215228
std           3.005983
min           0.000000
25%           8.000000
50%          10.000000
75%          12.000000
max          25.000000
Name: missing_count, dtype: float64

결측 개수 vs 지연 상관계수: 0.004


In [12]:
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

# X, y, groups는 지금 layout+조합피처가 들어간 상태여야 함
# (아까 9.187 나온 그 X 그대로)

lgb_params = {
    'n_estimators': 1000, 'learning_rate': 0.0100, 'num_leaves': 140,
    'max_depth': 8, 'min_child_samples': 25, 'subsample': 0.9608,
    'colsample_bytree': 0.6607, 'reg_alpha': 1.2338, 'reg_lambda': 7.6823,
    'random_state': 42, 'verbose': -1
}
xgb_params = {
    'n_estimators': 1000, 'learning_rate': 0.0101, 'max_depth': 7,
    'min_child_weight': 3, 'subsample': 0.6587, 'colsample_bytree': 0.8854,
    'reg_alpha': 3.3612, 'reg_lambda': 2.6707, 'random_state': 42
}

gkf = GroupKFold(n_splits=5)
lgb_maes, xgb_maes, rf_maes, ens_maes = [], [], [], []

for tr_idx, va_idx in gkf.split(X, y, groups=groups):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    y_tr_log = np.log1p(y_tr)

    m_lgb = LGBMRegressor(**lgb_params); m_lgb.fit(X_tr, y_tr_log)
    m_xgb = XGBRegressor(**xgb_params); m_xgb.fit(X_tr, y_tr_log)
    m_rf = RandomForestRegressor(n_estimators=200, max_depth=12,
                                 n_jobs=-1, random_state=42)
    m_rf.fit(X_tr, y_tr_log)

    p_lgb = np.expm1(m_lgb.predict(X_va))
    p_xgb = np.expm1(m_xgb.predict(X_va))
    p_rf  = np.expm1(m_rf.predict(X_va))
    p_ens = (p_lgb + p_xgb + p_rf) / 3

    lgb_maes.append(mean_absolute_error(y_va, p_lgb))
    xgb_maes.append(mean_absolute_error(y_va, p_xgb))
    rf_maes.append(mean_absolute_error(y_va, p_rf))
    ens_maes.append(mean_absolute_error(y_va, p_ens))

print(f'LGB:    {np.mean(lgb_maes):.3f}')
print(f'XGB:    {np.mean(xgb_maes):.3f}')
print(f'RF:     {np.mean(rf_maes):.3f}')
print(f'앙상블: {np.mean(ens_maes):.3f}')
print(f'(비교) LGB 단독 목표: 9.187')

LGB:    9.187
XGB:    9.190
RF:     9.230
앙상블: 9.172
(비교) LGB 단독 목표: 9.187
